In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Upload data from csv file



In [23]:
df = pd.read_csv("../data/insurance.csv")
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


##Clean and Encode Non-numerical Values



In [24]:
df_cleaned = df.dropna()
df_encoded = pd.get_dummies(df_cleaned, columns=[
    "sex",
    "smoker",
    "region",
], drop_first=True)

In [25]:
#store input columns in X
X = df_encoded[['age','bmi','children','sex_male','smoker_yes','region_northwest','region_southeast','region_southwest']]

In [26]:
#store output values in Y
Y = df_encoded["charges"]

##Split data into training and testing sections

In [27]:
x_train, x_test, y_train, y_test = train_test_split(X,Y, test_size=0.2, random_state=100)

##Create Random Forest Regressor model and fit training data

In [28]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,random_state=42
)
rf.fit(x_train,y_train)


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

##Use model to predict outcomes of training set and testing set

In [29]:
y_rf_train_pred = rf.predict(x_train)
y_rf_test_pred = rf.predict(x_test)

##Compare values in a dataframe showing difference

In [30]:
compare_outcomes = np.column_stack((y_train, y_rf_train_pred))
results = pd.DataFrame(compare_outcomes, columns=['actual_outcome','predicted_outcome'])
results['difference'] = abs(results['predicted_outcome'] - results['actual_outcome'])
results

,actual_outcome,predicted_outcome,difference
0,16115.30450,16369.317284,254.012784
1,10115.00885,10987.655699,872.646849
2,13635.63790,13485.238792,150.399108
3,5836.52040,6770.884360,934.363960
4,8871.15170,9701.126314,829.974614
...,...,...,...
1065,2103.08000,2341.979091,238.899091
1066,37742.57570,37963.725849,221.150149
1067,11830.60720,11983.700722,153.093522
1068,6571.02435,6456.807449,114.216901


##Calculate RMSE and r^2 values to determine accuracy of model

In [31]:
from sklearn.metrics import mean_squared_error, r2_score

rf_train_rmse = np.sqrt(mean_squared_error(y_train, y_rf_train_pred))
rf_train_r2 = r2_score(y_train, y_rf_train_pred)

rf_test_rmse = np.sqrt(mean_squared_error(y_test, y_rf_test_pred))
rf_test_r2 = r2_score(y_test, y_rf_test_pred)

##Store statistical results in dataframe

In [32]:
result = pd.DataFrame([rf_train_rmse, rf_train_r2, rf_test_rmse, rf_test_r2]).transpose()
result.columns = ["Training RMSE", "Training R^2", "Testing RMSE", "Testing R^2"]
result

,Training RMSE,Training R^2,Testing RMSE,Testing R^2
0,3121.693502,0.932214,4119.017846,0.891802


###Give new data to make a prediction

In [33]:
new_data = pd.read_csv("../data/new_data.csv")

##Predict value of charges for new data and export to CSV file

In [34]:
predicted_charges = rf.predict(new_data)

new_data['predicted_charges'] = predicted_charges
new_data.to_csv("../data/predicted_charges.csv", index=False)

In [35]:
new_data

,age,bmi,children,sex_male,smoker_yes,region_northwest,region_southeast,region_southwest,predicted_charges
0,23,27.5,0,1,0,0,1,0,4157.520258
1,45,28.1,2,0,1,1,0,0,25059.980990
2,31,22.8,1,1,0,0,0,1,5286.121461
3,54,35.6,3,0,1,0,1,0,45010.462243
4,19,24.1,0,1,0,1,0,0,1653.054521
5,37,29.4,2,0,0,0,0,1,7119.799326
6,62,33.8,1,1,1,0,1,0,47400.588307
7,28,26.3,0,0,0,1,0,0,3961.037169
8,41,31.9,2,1,0,0,1,0,8472.377751
9,50,28.7,3,0,1,0,0,1,25286.176276
